In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

In [2]:
#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [3]:
########################################
#FUNCTIONS

In [4]:
#JOB ARRAY SETUP

def StartJobArray(job_id):
    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel1['xh']) #total num of variables
    
    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    # job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    # if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id)
    print(f'\nstart_job = {start_job}, end_job = {end_job}')

    index_adjust=start_job
    return start_job,end_job,index_adjust
# job_id=1
# [start_job,end_job,index_adjust] = StartJobArray(job_id)

In [5]:
def GetData(start_job,end_job):
    # parcel=parcel1.isel(xh=slice(start_job,end_job))
    
    dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
    in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(in_file, 'r') as f:
        # Load the dataset by its name
        W = f['W'][:,start_job:end_job]
        Z = f['Z'][:,start_job:end_job]
        Y = f['Y'][:,start_job:end_job]
        X = f['X'][:,start_job:end_job]
        parcel_z=f['z'][:,start_job:end_job]

    return W,Z,Y,X,parcel_z
# [W,Z,Y,X,parcel_z] = GetData(start_job,end_job)    

In [6]:
#################################
#MORE FUNCTIONS

In [35]:
# Reading Back Data Later
##################################################################
#DOMAIN SUBSETTING
def DOMAIN_SUBSET(out_arr,index_adjust):
    print(f'length before: {len(out_arr)}')

    #SETTING UP
    ###########
    ocean_percent=2/8
    left_to_coast=data1['xh'][0]+(data1['xh'][-1]-data1['xh'][0])*ocean_percent
    
    where_coast_xh=np.where(data1['xh']>=left_to_coast)[0][0]#-25
    where_coast_xf=np.where(data1['xf']>=left_to_coast)[0][0]#-25
    end_xh=len(data1['xh'])-1-50
    end_xf=len(data1['xf'])-1-50
    
    print(f'x in {0}:{where_coast_xh-1} FOR SEA')
    print(f'x in {where_coast_xh}:{end_xh} FOR LAND')

    if res=='1km':
        t_start=36 
    elif res=='250m':
        t_start=36*5 
    t_end=len(data1['time']) 
    print(f't in {t_start}:end (8 hours)')

    #SUBSETTING CODE
    ################
    t,p=out_arr[:,1],out_arr[:,0]
    # if 'job_array' in globals():
    #     p -= index_adjust

    #GETTING X VALUES OF EACH PARCEL 
    xs=X[t,p-index_adjust]
    # fancy_index=True
    # # fancy_index=False
    # if fancy_index==True:
    #     xs=X[t,p] #FANCY INDEXING
    # elif fancy_index==False: #(slightly longer, but avoids loading X into memory)
    #     dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
    #     in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
    #     with h5py.File(in_file, 'r') as f:
    #         xs=[]
    #         for i, j in zip(t, p):
    #             xs.append(f['X'][i, j])
    #         xs=np.array(xs)
    ################
    
    out_arr=out_arr[np.where((xs>=where_coast_xh)&(xs<=end_xh))]
    out_arr=out_arr[np.where(out_arr[:,1]<=t_end)]

    print(f'==> length after: {len(out_arr)}'+'\n')
    return out_arr

In [36]:
def GetALLArrays_CL(start_job,end_job,index_adjust):
    #LOADING BACK IN
    def load_file():
        in_file=dir+f'Project_Algorithms/Tracking_Algorithms/parcel_tracking_{res}_{t_res}_{Np_str}.h5'
        with h5py.File(in_file, 'r') as hf:
            out_arr=hf['out_arr'][:]
            save_arr=hf['save_arr'][:]
            save2_arr=hf['save2_arr'][:]
        return out_arr,save_arr,save2_arr
    [out_arr,save_arr,save2_arr]=load_file()
    
    print('list of first 10 ignored parcels');
    # print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels'+'\n')
    
    # if 'job_array' in globals():
    #APPLYING JOB_ARRAY TO PARCEL NUMBER
    ####################################
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    print('Applying Job Array')
    out_arr=job_filter(out_arr)
    save_arr=job_filter(save_arr)
    
    # print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels'+'\n')
    
    #CHOOSING UNIQUE INDEXES
    ###############################################################################
    def remove_duplicates(arr):
        lst = []
        unique_values, counts = np.unique(arr[:, 0], return_counts=True)
        duplicates = unique_values[counts > 1]
        for elem in duplicates:
            idx = np.where(arr[:, 0] == elem)[0]
            extras = idx[np.where(arr[idx, 1] != np.min(arr[idx, 1]))]
            lst.extend(extras)
        mask = np.ones(len(arr), dtype=bool)
        mask[lst] = False
        return arr[mask]
    out_arr=remove_duplicates(out_arr)
    save_arr=remove_duplicates(save_arr)
    ###############################################################################
    
    ############################################################ 
    #SUBSETTING
    subset=True
    if subset==True:
        out_arr=DOMAIN_SUBSET(out_arr,index_adjust)
        save_arr=DOMAIN_SUBSET(save_arr,index_adjust)
    ############################################################
    ALL_out_arr=out_arr.copy(); ALL_save_arr=save_arr.copy()
    return ALL_out_arr,ALL_save_arr
# [ALL_out_arr,ALL_save_arr]=GetALLArrays_CL(start_job,end_job,index_adjust)

In [37]:
def ddt(f,dt=1):
    ddx = (
            f[1:  ]
            -
            f[0:-1]
        ) / (
        2 * dt
    )
    return ddx

#search for deep convective parcels within lagrangian tracking output     
##############################################################
def SHALLOW_threshold(out_arr,zthresh,index_adjust,type):

    if type=='CL':
        out_arr=ALL_out_arr.copy()
    elif type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data1['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        # if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 

        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data1['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY ADJUST
    
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        elif np.any(above[local_maxes[0]]<=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

In [38]:
#search for deep convective parcels within lagrangian tracking output     
##############################################################
def DEEP_threshold(out_arr,zthresh,index_adjust,type):
    if type=='CL':
        out_arr=ALL_out_arr.copy()
    elif type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data1['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        # if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 

        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data1['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY ADJUST
        
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]>=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

In [39]:
#SHALLOW
def GetSHALLOWArrays_CL(ALL_out_arr,ALL_save_arr,index_adjust):
    convectivelevel=4 #4km
    SHALLOW_out_arr=SHALLOW_threshold(ALL_out_arr,convectivelevel,index_adjust,type='CL')
    SHALLOW_save_arr=SHALLOW_threshold(ALL_save_arr,convectivelevel,index_adjust,type='nonCL')
    
    # print('list of first 10 SBZ parcels'); print(out_arr[:15])
    # print(f'there are a total of {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')
    return SHALLOW_out_arr,SHALLOW_save_arr
# [SHALLOW_out_arr,SHALLOW_save_arr]=GetSHALLOWArrays_CL(ALL_out_arr,ALL_save_arr)

In [40]:
#DEEP
def GetDEEPArrays_CL(ALL_out_arr,ALL_save_arr,index_adjust):    
    convectivelevel=6 #4km
    DEEP_out_arr=DEEP_threshold(ALL_out_arr,convectivelevel,index_adjust,type='CL')
    DEEP_save_arr=DEEP_threshold(ALL_save_arr,convectivelevel,index_adjust,type='nonCL')
    
    # print('list of first 10 SBZ parcels'); print(out_arr[:15])
    # print(f'there are a total of {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')
    return DEEP_out_arr,DEEP_save_arr
# [DEEP_out_arr,DEEP_save_arr]=GetDEEPArrays_CL(ALL_out_arr,ALL_save_arr)

In [41]:
def GetArrays_SBZ(ALL_out_arr,index_adjust):
    
    
    def find_SBZ_xmaxs():
        # Define the directory and file path
        dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
        file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{t_res}' + '.h5'
        
        # Open the HDF5 file in read mode
        with h5py.File(file_path, 'r') as f:
            # Access the 'conv' dataset
            conv_dataset = f['conv']
            
            # Define the vertical level you are interested in
            zlev = 4
            
            # Initialize a list to store the xmaxs for each time step
            xmaxs_list = []
    
            # Loop over each time step (axis=0 corresponds to time)
            for t in range(conv_dataset.shape[0]):  # conv_dataset.shape[0] is the time dimension size
                # Read the relevant slice for this time step and vertical level
                Conv_t_zlev = conv_dataset[t, zlev, :, :]  # Shape should be (y_size, x_size)
                
                # Calculate the mean across the y-axis
                Conv_ymean = np.mean(Conv_t_zlev, axis=0)  # Mean across the y-axis
                
                # Find the index of the maximum value along the x-axis
                xmax = np.argmax(Conv_ymean)
                
                # Append the result for this time step
                xmaxs_list.append(xmax)
        
        # Convert the list of xmaxs to a numpy array (optional)
        xmaxs = np.array(xmaxs_list)
    
        return xmaxs #returns SBZ x location for each timestep
    
    
    def subset_SBZ(out_arr):
        xmaxs=find_SBZ_xmaxs()
    
        SBZ_subset=[]
        # test=[] #TESTING
        
        for ind in np.arange(out_arr.shape[0]):
            
            row=out_arr[ind]
            p=row[0]
            t=row[1]
    
            kms=np.argmax(data1['xh'].values-data1['xh'][0].values >= 1)
            if X[t,p-index_adjust] in np.arange( (xmaxs[t]-2*kms),(xmaxs[t]+2*kms) +1):
                SBZ_subset.append(ind)
                # test.append(p) #TESTING
        
        SBZ_out_arr=out_arr[SBZ_subset]
        print(f'there are a total of {len(SBZ_out_arr)} ALL SBZ CL parcels')
    
        valid_range=np.arange(out_arr.shape[0])
        nonSBZ_out_arr=out_arr[list(set(valid_range) - set(SBZ_subset))]
        print(f'there are a total of {len(nonSBZ_out_arr)} ALL nonSBZ CL parcels')
        return SBZ_out_arr,nonSBZ_out_arr
    
    
    # #LOADING CL MAXS FROM CL TRACKING ALGORITHM
    # folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/'
    # whereSBZ=xr.open_dataset(folder+f'whereCL_{res}_ONLY_SBZS.nc').load()
    # whereSBZ=whereSBZ.isel(time=slice(0,len(data1['time'])))
    # whereSBZ=whereSBZ['maxconv_x']
    # def Get_SBZ_X(t,z,y):
    #     Conv_X_Max=whereSBZ[t,z,y,:].values
    #     return Conv_X_Max
    # def subset_SBZ(out_arr):
    
    #     SBZ_subset=[]
    #     # test=[] #TESTING
        
    #     for ind in np.arange(out_arr.shape[0]):
            
    #         row=out_arr[ind]
    #         p=row[0]
    #         t=row[1]
    
    #         kms=np.argmax(data1['xh'].values-data1['xh'][0].values >= 1)
    #         value=X[t,p]
    #         if np.any((value >= xmaxs - 2*kms) & (value <= xmaxs + 2*kms))==True:
    #             SBZ_subset.append(ind)
    #             # test.append(p) #TESTING
        
    #     SBZ_out_arr=out_arr[SBZ_subset]
    #     print(f'there are a total of {len(SBZ_out_arr)} ALL SBZ CL parcels')
    
    #     valid_range=np.arange(out_arr.shape[0])
    #     nonSBZ_out_arr=out_arr[list(set(valid_range) - set(SBZ_subset))]
    #     print(f'there are a total of {len(nonSBZ_out_arr)} ALL nonSBZ CL parcels')
    #     return SBZ_out_arr,nonSBZ_out_arr
    
    
    # print(f'there are a total of {len(SHALLOW_SBZ_out_arr)} SHALLOW SBZ CL parcels')
    # print(f'there are a total of {len(SHALLOW_nonSBZ_out_arr)} SHALLOW nonSBZ CL parcels')
    # print(f'there are a total of {len(DEEP_SBZ_out_arr)} DEEP SBZ CL parcels')
    # print(f'there are a total of {len(DEEP_nonSBZ_out_arr)} DEEP nonSBZ CL parcels')
    
    #SUBSETTING OUT SHALLOW AND DEEP FROM SBZ AND NONSBZ
    [ALL_SBZ_out_arr,ALL_nonSBZ_out_arr]=subset_SBZ(ALL_out_arr)
    SHALLOW_SBZ_out_arr=SHALLOW_threshold(ALL_SBZ_out_arr,4,index_adjust,'SBZ')
    SHALLOW_nonSBZ_out_arr=SHALLOW_threshold(ALL_nonSBZ_out_arr,4,index_adjust,'nonSBZ')
    DEEP_SBZ_out_arr=DEEP_threshold(ALL_SBZ_out_arr,6,index_adjust,'SBZ')
    DEEP_nonSBZ_out_arr=DEEP_threshold(ALL_nonSBZ_out_arr,6,index_adjust,'nonSBZ')

    return ALL_SBZ_out_arr,ALL_nonSBZ_out_arr,SHALLOW_SBZ_out_arr,SHALLOW_nonSBZ_out_arr,DEEP_SBZ_out_arr,DEEP_nonSBZ_out_arr


In [42]:
def get_ColdPool(out_arr1,out_arr2):
    arr1 = out_arr1[:,0] #CL
    arr2 = out_arr2[:,0] #nonSBZ
    common_values = np.intersect1d(arr1, arr2)
    indices_arr1 = np.where(np.isin(arr1, common_values))[0]  # Indices in arr1
    ColdPool_out_arr=out_arr1[indices_arr1]
    return ColdPool_out_arr

In [43]:
def SaveData(data_dict, job_id):
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/subsetting/'
    out_file = dir2 + f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}_{job_id}"

    # Write the data to HDF5 file
    with h5py.File(out_file, 'w') as h5f:
        for key, value in data_dict.items():
            h5f.create_dataset(key, data=value)

In [44]:
counts=[[],[],[],[],[],[]]
def AddCounts(counts,ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr, job_id):
    counts[0].append(ALL_out_arr.shape[0])
    counts[1].append(SHALLOW_out_arr.shape[0])
    counts[2].append(DEEP_out_arr.shape[0])
    
    counts[3].append(ALL_save_arr.shape[0])
    counts[4].append(SHALLOW_save_arr.shape[0])
    counts[5].append(DEEP_save_arr.shape[0])
    return counts
# AddCounts(counts,ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr, job_id)

In [45]:
#####################################
#RUNNING

In [46]:
count_dict = {key: [] for key in [
    'ALL_out_arr', 'SHALLOW_out_arr', 'DEEP_out_arr',
    'ALL_save_arr', 'SHALLOW_save_arr', 'DEEP_save_arr',
    
    'ALL_SBZ_out_arr', 'ALL_nonSBZ_out_arr',
    'SHALLOW_SBZ_out_arr', 'SHALLOW_nonSBZ_out_arr',
    'DEEP_SBZ_out_arr', 'DEEP_nonSBZ_out_arr',
    
    'ALL_ColdPool_out_arr', 'SHALLOW_ColdPool_out_arr',
    'DEEP_ColdPool_out_arr'
]}

In [47]:
num_jobs = 60
for job_id in np.arange(1, num_jobs + 1):
    if job_id % 10: print(f"current job_id: {job_id}")
    
    [start_job, end_job, index_adjust] = StartJobArray(job_id)
    [W, Z, Y, X, parcel_z] = GetData(start_job, end_job) 

    #CL vs nonCL
    [ALL_out_arr, ALL_save_arr] = GetALLArrays_CL(start_job,end_job,index_adjust)
    [SHALLOW_out_arr, SHALLOW_save_arr] = GetSHALLOWArrays_CL(ALL_out_arr, ALL_save_arr,index_adjust)
    [DEEP_out_arr, DEEP_save_arr] = GetDEEPArrays_CL(ALL_out_arr, ALL_save_arr,index_adjust)

    #SBZ vs nonSBZ
    [ALL_SBZ_out_arr, ALL_nonSBZ_out_arr, SHALLOW_SBZ_out_arr,
     SHALLOW_nonSBZ_out_arr, DEEP_SBZ_out_arr, DEEP_nonSBZ_out_arr] = GetArrays_SBZ(ALL_out_arr, index_adjust)

    #COLDPOOL
    ALL_ColdPool_out_arr = get_ColdPool(ALL_out_arr, ALL_nonSBZ_out_arr)
    SHALLOW_ColdPool_out_arr = get_ColdPool(SHALLOW_out_arr, SHALLOW_nonSBZ_out_arr)
    DEEP_ColdPool_out_arr = get_ColdPool(DEEP_out_arr, DEEP_nonSBZ_out_arr)

    # Create a dictionary of arrays to save (including SBZ arrays)
    data_dict = {
        'ALL_out_arr': ALL_out_arr,
        'SHALLOW_out_arr': SHALLOW_out_arr,
        'DEEP_out_arr': DEEP_out_arr,
        'ALL_save_arr': ALL_save_arr,
        'SHALLOW_save_arr': SHALLOW_save_arr,
        'DEEP_save_arr': DEEP_save_arr,
        
        'ALL_SBZ_out_arr': ALL_SBZ_out_arr,
        'ALL_nonSBZ_out_arr': ALL_nonSBZ_out_arr,
        'SHALLOW_SBZ_out_arr': SHALLOW_SBZ_out_arr,
        'SHALLOW_nonSBZ_out_arr': SHALLOW_nonSBZ_out_arr,
        'DEEP_SBZ_out_arr': DEEP_SBZ_out_arr,
        'DEEP_nonSBZ_out_arr': DEEP_nonSBZ_out_arr,
        
        'ALL_ColdPool_out_arr': ALL_ColdPool_out_arr,
        'SHALLOW_ColdPool_out_arr': SHALLOW_ColdPool_out_arr,
        'DEEP_ColdPool_out_arr': DEEP_ColdPool_out_arr
    }
    for key in count_dict:
        count_dict[key].append(data_dict[key].shape[0])
    
    # Call SaveData with the dictionary
    SaveData(data_dict, job_id)

current job_id: 1

start_job = 0, end_job = 16667
list of first 10 ignored parcels
Applying Job Array
length before: 232
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 231

length before: 238
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 178

there are a total of 8 ALL SBZ CL parcels
there are a total of 223 ALL nonSBZ CL parcels
current job_id: 2

start_job = 16667, end_job = 33334
list of first 10 ignored parcels
Applying Job Array
length before: 251
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 251

length before: 248
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 199

there are a total of 10 ALL SBZ CL parcels
there are a total of 241 ALL nonSBZ CL parcels
current job_id: 3

start_job = 33334, end_job = 50001
list of first 10 ignored parcels
Applying Job Array
length before: 227
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> l

In [49]:
#GETTING COUNTS FOR MAKING INITIAL RECOMBINED ARRAYS LATER
combined_counts={key: sum(counts) for key, counts in count_dict.items()}
print(combined_counts)

{'ALL_out_arr': 14630, 'SHALLOW_out_arr': 10059, 'DEEP_out_arr': 1489, 'ALL_save_arr': 10991, 'SHALLOW_save_arr': 8367, 'DEEP_save_arr': 1112, 'ALL_SBZ_out_arr': 815, 'ALL_nonSBZ_out_arr': 13815, 'SHALLOW_SBZ_out_arr': 424, 'SHALLOW_nonSBZ_out_arr': 9635, 'DEEP_SBZ_out_arr': 187, 'DEEP_nonSBZ_out_arr': 1302, 'ALL_ColdPool_out_arr': 13815, 'SHALLOW_ColdPool_out_arr': 9635, 'DEEP_ColdPool_out_arr': 1302}


In [50]:
######################################
#RECOMBINING

In [51]:
def ReadData(var_name,job_id):
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/subsetting/'
    out_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}_{job_id}"
    with h5py.File(out_file, 'r') as f:
        out = f[var_name][:]
    return out
# ReadData(var_name,job_id)

In [52]:
def MakeEmpty(counts_dict):
    empty_dict = {}
    for key, count in counts_dict.items():
        empty_dict[key] = np.zeros((count, 3), dtype=int)
    return empty_dict
dict=MakeEmpty(combined_counts)

In [53]:
num_jobs=60

for job_id in np.arange(1,num_jobs+1):
    if job_id % 10==0: print(f"current job_id: {job_id}")
        
    for key in dict:
        var=ReadData(key,job_id)
        if var.size!=0:
            a=np.where(np.all(dict[key] == 0, axis=1))[0][0]
            b=a+var.shape[0]
            # print(key,a,b) #TESTING
            dict[key][a:b]=var

current job_id: 10
current job_id: 20
current job_id: 30
current job_id: 40
current job_id: 50
current job_id: 60


In [54]:
def SaveFinalData(dict,out_file):    
    with h5py.File(out_file,'w') as f:
        for key in dict:
            print(key)
            f.create_dataset(key, data=dict[key])

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
out_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
SaveFinalData(dict,out_file)

ALL_out_arr
SHALLOW_out_arr
DEEP_out_arr
ALL_save_arr
SHALLOW_save_arr
DEEP_save_arr
ALL_SBZ_out_arr
ALL_nonSBZ_out_arr
SHALLOW_SBZ_out_arr
SHALLOW_nonSBZ_out_arr
DEEP_SBZ_out_arr
DEEP_nonSBZ_out_arr
ALL_ColdPool_out_arr
SHALLOW_ColdPool_out_arr
DEEP_ColdPool_out_arr


In [55]:
########################
#READING BACK IN

In [56]:
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

#PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(ALL_out_arr)} CL parcels and {len(ALL_save_arr)} nonCL parcels')
print(f'SHALLOW: {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')
print(f'DEEP: {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(ALL_SBZ_out_arr)} SBZ parcels and {len(ALL_nonSBZ_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SHALLOW_SBZ_out_arr)} SBZ parcels and {len(SHALLOW_nonSBZ_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(DEEP_SBZ_out_arr)} SBZ parcels and {len(DEEP_nonSBZ_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ALL_ColdPool_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(SHALLOW_ColdPool_out_arr)} ColdPool parcels')
print(f'DEEP: {len(DEEP_ColdPool_out_arr)} ColdPool parcels')

ALL: 14630 CL parcels and 10991 nonCL parcels
SHALLOW: 10059 CL parcels and 8367 nonCL parcels
DEEP: 1489 CL parcels and 1112 nonCL parcels


ALL: 815 SBZ parcels and 13815 nonSBZ parcels
SHALLOW: 424 SBZ parcels and 9635 nonSBZ parcels
DEEP: 187 SBZ parcels and 1302 nonSBZ parcels


ALL: 13815 ColdPool parcels
SHALLOW: 9635 ColdPool parcels
DEEP: 1302 ColdPool parcels


In [ ]:
#######################################
#OLD TESTING THINGS

In [324]:
# #TESTING 
# in_file=dir+f'Project_Algorithms/Tracking_Algorithms/parcel_tracking_{res}_{t_res}_{Np_str}.h5'
# def load_file():
#     with h5py.File(in_file, 'r') as hf:
#         out_arr=hf['out_arr'][:]
#         save_arr=hf['save_arr'][:]
#         save2_arr=hf['save2_arr'][:]
#     return out_arr,save_arr,save2_arr
# [out_arr,save_arr,save2_arr]=load_file()
# hey=dict['ALL_out_arr']
# arr=out_arr.copy()
# np.all(arr==hey)

In [325]:
#TESTING
# len(out_arr[:,0])-len(np.unique(out_arr[:,0]))
# arr=out_arr.copy()
# # Step 1: Use np.unique with return_counts and return_inverse
# unique_rows, indices, counts = np.unique(arr, axis=0, return_inverse=True, return_counts=True)
# plt.plot(counts)

In [326]:
#TESTING
# dict['ALL_out_arr']

# hey=dict['ALL_out_arr']
# np.where(np.all(hey==0,axis=1))

# plt.plot(hey[:,0])
# plt.plot(out_arr[:,0])

In [ ]:
#################
#OLD THINGS

In [ ]:
# def SaveData(ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr, job_id):
#     dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/subsetting/'
#     out_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}_{job_id}"
#     with h5py.File(out_file, 'w') as h5f:
#         h5f.create_dataset('ALL_out_arr', data=ALL_out_arr)
#         h5f.create_dataset('SHALLOW_out_arr', data=SHALLOW_out_arr)
#         h5f.create_dataset('DEEP_out_arr', data=DEEP_out_arr)

#         h5f.create_dataset('ALL_save_arr', data=ALL_save_arr)
#         h5f.create_dataset('SHALLOW_save_arr', data=SHALLOW_save_arr)
#         h5f.create_dataset('DEEP_save_arr', data=DEEP_save_arr)
# # SaveData(ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr, job_id)

In [ ]:
# num_jobs=60
# for job_id in np.arange(1,num_jobs+1):
#     if job_id % 10: print(f"current job_id: {job_id}")
#     [start_job,end_job,index_adjust] = StartJobArray(job_id)
#     [W,Z,Y,X,parcel_z] = GetData(start_job,end_job) 
#     [ALL_out_arr,ALL_save_arr]=GetALLArrays_CL(start_job,end_job,index_adjust)
#     [SHALLOW_out_arr,SHALLOW_save_arr]=GetSHALLOWArrays_CL(ALL_out_arr,ALL_save_arr)
#     [DEEP_out_arr,DEEP_save_arr]=GetDEEPArrays_CL(ALL_out_arr,ALL_save_arr)
#     [ALL_SBZ_out_arr,ALL_nonSBZ_out_arr,SHALLOW_SBZ_out_arr,
#      SHALLOW_nonSBZ_out_arr,DEEP_SBZ_out_arr,DEEP_nonSBZ_out_arr]=GetArrays_SBZ(ALL_out_arr,index_adjust)

#     #*****
#     AddCounts(counts,ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr, job_id)
#     SaveData(ALL_out_arr,SHALLOW_out_arr,DEEP_out_arr,
#              ALL_save_arr,SHALLOW_save_arr,DEEP_save_arr,
             
#              job_id)